In [20]:
from qdrant_client import QdrantClient                                                                                                         
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams                                                 
                                                                                                                                                
try:                                                                                                                                           
    client = QdrantClient(url="http://127.0.0.1:6333", timeout=5, check_compatibility=False)                                                   
    client.get_collections()                                                                                                                   
    print("🎉 Successfully connected to Qdrant Docker on port 6333.")                                                                          
except Exception as e:                                                                                                                         
    print("Could not connect to local Docker Qdrant. Falling back to local in-memory instance.")                                               
    print(f"Error details: {e}")                                                                                                               
    client = QdrantClient(":memory:")                                                                                                          
                                                                                                                                                
COLLECTION_NAME = "cuad_advanced"

🎉 Successfully connected to Qdrant Docker on port 6333.


In [21]:
# Recreate the collection (delete if exists, then create new)
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense-bge": VectorParams(
            size=384,  # Dimension of BAAI/bge-small-en-v1.5
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "sparse-bm25": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=True
            )
        )
    }
)

print(f"Collection '{COLLECTION_NAME}' created and configured successfully for Hybrid Search!")

Collection 'cuad_advanced' created and configured successfully for Hybrid Search!


In [22]:
from qdrant_client.models import PayloadSchemaType

print("Creating payload indexes for metadata filter....")

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="year",
    field_schema=PayloadSchemaType.KEYWORD
)
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="section",
    field_schema=PayloadSchemaType.KEYWORD
)

print("Payload indexes established successfully!")

Creating payload indexes for metadata filter....
Payload indexes established successfully!


In [23]:
from fastembed import TextEmbedding, SparseTextEmbedding

print("Loading Dense Embedding Model (BAAI/bge-small-en-v1.5)...")
dense_model = TextEmbedding("BAAI/bge-small-en-v1.5")

print("Loading Sparse BM25 Model (Qdrant/bm25)...")
sparse_model = SparseTextEmbedding("Qdrant/bm25")

print("FastEmbed models loaded and ready!")

Loading Dense Embedding Model (BAAI/bge-small-en-v1.5)...
Loading Sparse BM25 Model (Qdrant/bm25)...
FastEmbed models loaded and ready!


In [24]:
sample_parsed_chunks = [
    {
        "chunk_id": "c1-uuid",
        "chunk_text": "LIMITATION OF LIABILITY. Except for indemnity obligations under Article 8 or breach of confidentiality duties, in no event shall either party's aggregate liability under this Credit Agreement exceed $10,000,000. Neither party shall be liable for indirect, special, punitive, or consequential damages.",
        "contract_name": "Applied Materials Credit Agreement",
        "file_name": "000000000.txt",
        "year": "2000",
        "section": "SECTION 10.04. Limitation of Liability"
    },
    {
        "chunk_id": "c2-uuid",
        "chunk_text": "INDEMNIFICATION BY BORROWER. The Borrower agrees to indemnify, defend, and hold harmless the Agent and the Lenders from and against any losses, claims, damages, liabilities, and expenses arising out of the transactions contemplated by this 364-day Credit Agreement, except to the extent resulting from gross negligence.",
        "contract_name": "Applied Materials Credit Agreement",
        "file_name": "000000000.txt",
        "year": "2000",
        "section": "SECTION 9.03. Indemnification"
    },
    {
        "chunk_id": "c3-uuid",
        "chunk_text": "GOVERNING LAW. This Credit Agreement shall be governed by, and construed in accordance with, the laws of the State of New York without regard to conflict of law principles. Any legal action arising hereunder shall be brought in the federal courts located in the Southern District of New York.",
        "contract_name": "Applied Materials Credit Agreement",
        "file_name": "000000000.txt",
        "year": "2000",
        "section": "SECTION 10.09. Governing Law"
    }
]

print(f"Loaded {len(sample_parsed_chunks)} chunks for indexing.")

Loaded 3 chunks for indexing.


In [25]:
from qdrant_client.models import PointStruct,SparseVector
chunks_texts=[item["chunk_text"]for item in sample_parsed_chunks]
dense_embeddings=list(dense_model.embed(chunks_texts))
sparse_embeddings=list(sparse_model.embed(chunks_texts))
points=[]

for idx, item in enumerate(sample_parsed_chunks):
    sparse_obj = sparse_embeddings[idx]
    qdrant_sparse = SparseVector(
        indices=sparse_obj.indices.tolist(),
        values=sparse_obj.values.tolist()
    )
    
    point = PointStruct(
        id=idx + 1,
        vector={
            "dense-bge": dense_embeddings[idx].tolist(),
            "sparse-bm25": qdrant_sparse
        },
        payload={
            "chunk_id": item["chunk_id"],
            "chunk_text": item["chunk_text"],
            "contract_name": item["contract_name"],
            "file_name": item["file_name"],
            "year": item["year"],
            "section": item["section"]
        }
    )
    points.append(point)

# 4. Upload
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print(f"Uploaded {len(points)} vectors to Qdrant successfully!")


Uploaded 3 vectors to Qdrant successfully!


In [26]:
query_text = "What is the liability cap under the Applied Materials contract?"

# 1. Generate query vectors
query_dense = list(dense_model.embed([query_text]))[0].tolist()
query_sparse_obj = list(sparse_model.embed([query_text]))[0]
query_sparse = SparseVector(
    indices=query_sparse_obj.indices.tolist(),
    values=query_sparse_obj.values.tolist()
)
# 2. Search Dense-Only
dense_response = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_dense,
    using="dense-bge",
    limit=2,
    with_payload=True,
)

print("=== DENSE SEARCH RESULTS ===")
for res in dense_response.points:
    print(
        f"Score: {res.score:.4f} | "
        f"Contract: {res.payload['contract_name']} | "
        f"Section: {res.payload['section']} | "
        f"Chunk: '{res.payload['chunk_text'][:80]}...'"
    )

# 3. Search Sparse-Only
sparse_response = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_sparse,
    using="sparse-bm25",
    limit=2,
    with_payload=True,
)

print("\n=== SPARSE SEARCH RESULTS ===")
for res in sparse_response.points:
    print(
        f"Score: {res.score:.4f} | "
        f"Contract: {res.payload['contract_name']} | "
        f"Section: {res.payload['section']} | "
        f"Chunk: '{res.payload['chunk_text'][:80]}...'"
    )

=== DENSE SEARCH RESULTS ===
Score: 0.7194 | Contract: Applied Materials Credit Agreement | Section: SECTION 10.04. Limitation of Liability | Chunk: 'LIMITATION OF LIABILITY. Except for indemnity obligations under Article 8 or bre...'
Score: 0.6457 | Contract: Applied Materials Credit Agreement | Section: SECTION 9.03. Indemnification | Chunk: 'INDEMNIFICATION BY BORROWER. The Borrower agrees to indemnify, defend, and hold ...'

=== SPARSE SEARCH RESULTS ===
Score: 3.0498 | Contract: Applied Materials Credit Agreement | Section: SECTION 10.04. Limitation of Liability | Chunk: 'LIMITATION OF LIABILITY. Except for indemnity obligations under Article 8 or bre...'
Score: 2.6334 | Contract: Applied Materials Credit Agreement | Section: SECTION 9.03. Indemnification | Chunk: 'INDEMNIFICATION BY BORROWER. The Borrower agrees to indemnify, defend, and hold ...'
